<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-05-tree-models/05_Tree_Ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup, W&B & Load Wine Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_wine
import wandb

# Inisialisasi W&B
run = wandb.init(project="wine-classification-trees", name="tree-vs-forest")

# Load Dataset Wine (Klasifikasi 3 jenis anggur berdasarkan tes kimia)
data = load_wine()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target # Terdapat 3 kelas: 0, 1, 2

# Log raw data
wandb.log({"raw_data": wandb.Table(dataframe=df.head())})
print("Dataset Shape:", df.shape)
df.head()

Split Data & Train Decision Tree Tunggal

In [ ]:
# Split Data
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Melatih Decision Tree Tunggal
dt_model = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_model.fit(X_train, y_train)

# Evaluasi
y_pred_dt = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy Decision Tree: {acc_dt:.4f}")

Visualisasi Pohon Keputusan

In [ ]:
# Melihat bagaimana AI mengambil keputusan secara visual
plt.figure(figsize=(20, 10))
plot_tree(dt_model,
          feature_names=data.feature_names,
          class_names=data.target_names,
          filled=True, rounded=True, fontsize=12)
plt.title("Visualisasi Pembelahan Decision Tree")
plt.show()

Train Random Forest

In [ ]:
# 2. Melatih Random Forest (Membuat 100 pohon secara bersamaan)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluasi
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

wandb.log({
    "Accuracy_DecisionTree": acc_dt,
    "Accuracy_RandomForest": acc_rf
})

print(f"Accuracy Random Forest: {acc_rf:.4f}")
print("\nClassification Report Random Forest:\n", classification_report(y_test, y_pred_rf))

Analisis Tingkat Kepentingan Fitur

In [ ]:
# Mencari tahu fitur medis/kimia mana yang paling berpengaruh pada hasil akhir
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1] # Urutkan dari yang paling penting

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=np.array(data.feature_names)[indices], palette="viridis", hue=np.array(data.feature_names)[indices], legend=False)
plt.title("Feature Importances in Random Forest (Wine Dataset)")
plt.xlabel("Tingkat Kepentingan (0 - 1)")
plt.ylabel("Fitur Kimia")
plt.tight_layout()
plt.show()

wandb.finish()